# Instruct vs No-Finetuning Ablation — Combined Results

Compares **pretrained-only** ("no-FT") and **instruction-tuned** ("instruct") variants of the same base
model across CI benchmarks, from the **`eval-all`** W&B project. For each model, shows the delta
from no-FT → instruct, testing whether instruction tuning meaningfully shifts contextual
privacy-reasoning performance.

**Benchmarks**: GoldCoin-HIPAA (applicability + compliance), PrivacyLens (QA probing + leakage + helpful),
ConfAIde (Pearson r), CIRL-Vignettes (completeness), VLM-GeoPrivacy (Q7).

**Model pairs**: Qwen3.5-2B, Qwen3.5-4B, Qwen3.5-9B (pretrained vs instruct).

> Note: Qwen3.5-9B-Base currently has partial coverage (goldcoin_hipaa + privacylens only) —
> missing cells render as `---`. Re-fetch the W&B cache to pull newer runs.

In [ ]:
import subprocess, sys
from pathlib import Path

# Resolve this notebook's directory for local imports (wandb_cache.py)
if "__vsc_ipynb_file__" in globals():
    _NB_DIR = str(Path(globals()["__vsc_ipynb_file__"]).parent)
elif (Path.cwd() / "wandb_cache.py").exists():
    _NB_DIR = str(Path.cwd())
else:
    _repo = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
    _NB_DIR = str(Path(_repo) / "notebooks" / "wicked")
if _NB_DIR not in sys.path:
    sys.path.insert(0, _NB_DIR)

from wandb_cache import load_runs, cache_info
import importlib, wandb_cache; importlib.reload(wandb_cache)
from wandb_cache import load_runs, cache_info
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Latex

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:.3f}".format)

# ── Configuration ──────────────────────────────────────────────────────────
# Map: pretrained checkpoint name  →  instruct checkpoint name
# Both live under tags=["base"] in W&B; the "-Base" suffix marks pretrained.
BASE_TO_INSTRUCT = {
    "Qwen3.5-2B-Base": "Qwen3.5-2B",
    "Qwen3.5-4B-Base": "Qwen3.5-4B",
    "Qwen3.5-9B-Base": "Qwen3.5-9B",
}

# Display labels keyed by the *instruct* checkpoint name (the canonical model).
MODEL_DISPLAY = {
    "Qwen3.5-2B": "Qwen3.5-2B",
    "Qwen3.5-4B": "Qwen3.5-4B",
    "Qwen3.5-9B": "Qwen3.5-9B",
}
MODEL_ORDER = list(MODEL_DISPLAY.values())
BOLD_MODELS = set()  # no special highlighting for this ablation

# Dagspace short prefixes (used in pivot)
DS_PREFIX = {
    "goldcoin_hipaa": "gc", "privacylens": "pl",
    "vlm_geoprivacy_bench": "vlm", "confaide": "ca",
    "cirl_vignettes": "cirl",
}

info = cache_info()
print(f"Using cached W&B data from {info.get('fetched_at', 'unknown')} ({info.get('total_runs', '?')} runs)")


## 1. Load cached runs


In [ ]:
# Both conditions live under tags=["base"] — the "-Base" suffix on the checkpoint
# name distinguishes pretrained-only from instruction-tuned. One load call suffices.
all_df = load_runs(tags=["base"], has_metrics=True)
pretrained_names = set(BASE_TO_INSTRUCT.keys())
instruct_names = set(BASE_TO_INSTRUCT.values())
pretrained_df = all_df[all_df["checkpoint_name"].isin(pretrained_names)]
instruct_df = all_df[all_df["checkpoint_name"].isin(instruct_names)]
print(f"Found {len(pretrained_df)} pretrained (no-FT) runs, {len(instruct_df)} instruct runs")


## 2. Extract metrics into a unified DataFrame


In [ ]:
def _label_runs(df, condition, checkpoint_map=None):
    """Add model/condition columns; optionally remap pretrained → instruct name."""
    df = df.copy()
    df["checkpoint_raw"] = df["checkpoint_name"]
    if checkpoint_map:
        df["model"] = df["checkpoint_name"].map(checkpoint_map)
        df = df.dropna(subset=["model"])
    else:
        df["model"] = df["checkpoint_name"]
    df = df[df["model"].isin(MODEL_DISPLAY)]
    df["condition"] = condition
    return df

pretrained_labelled = _label_runs(pretrained_df, "no-ft", BASE_TO_INSTRUCT)
instruct_labelled = _label_runs(instruct_df, "instruct")

df_raw = pd.concat([pretrained_labelled, instruct_labelled], ignore_index=True)

# Identify all metric columns
precomputed_cols = [c for c in df_raw.columns if c.startswith(("gc_", "pl_", "vlm_"))]
eval_cols = sorted([c for c in df_raw.columns if c.startswith("eval/")])
all_metric_cols = precomputed_cols + eval_cols

print(f"Extracted {len(df_raw)} metric rows")
print(f"Models: {sorted(df_raw['model'].unique())}")
print(f"Conditions: {sorted(df_raw['condition'].unique())}")

# Coverage grid: (model × condition × dagspace) run counts
coverage = (df_raw.groupby(["model", "condition", "dagspace"]).size()
            .unstack(fill_value=0))
print("\nCoverage (run counts by model, condition, dagspace):")
display(coverage)

display(df_raw)


In [ ]:
df_raw['checkpoint_raw'].value_counts()

## 3. Pivot ALL metrics into per-model comparison table

One row per (model, condition). Metrics pivoted per dagspace with dagspace prefix.


In [ ]:
# De-duplicate: keep last run per (model, condition, dagspace)
df_dedup = df_raw.sort_values("created_at").drop_duplicates(
    subset=["model", "condition", "dagspace"], keep="last"
)

def _pivot_dagspace(df, dagspace):
    sub = df[df["dagspace"] == dagspace].copy()
    if sub.empty:
        return pd.DataFrame()
    prefix = DS_PREFIX.get(dagspace, dagspace)
    metric_cols = [c for c in all_metric_cols if sub[c].notna().any()]
    out = sub[["model", "condition"] + metric_cols]
    rename = {c: f"{prefix}/{c[5:]}" for c in out.columns if c.startswith("eval/")}
    return out.rename(columns=rename)

# Merge all dagspaces
df_table = pd.DataFrame()
for ds in sorted(df_dedup["dagspace"].unique()):
    piv = _pivot_dagspace(df_dedup, ds)
    if piv.empty:
        continue
    if df_table.empty:
        df_table = piv
    else:
        overlap = [c for c in piv.columns if c in df_table.columns and c not in ["model", "condition"]]
        if overlap:
            piv = piv.drop(columns=overlap)
        df_table = df_table.merge(piv, on=["model", "condition"], how="outer")

df_table["Model"] = df_table["model"].map(MODEL_DISPLAY).fillna(df_table["model"])

# Convert [0,1] to percentage
for c in df_table.select_dtypes(include="number").columns:
    df_table[c] = df_table[c] * 100

# Sort by model order, no-ft before instruct
condition_order = {"no-ft": 0, "instruct": 1}
model_order_map = {m: i for i, m in enumerate(MODEL_ORDER)}
df_table["_mr"] = df_table["Model"].map(model_order_map).fillna(99)
df_table["_cr"] = df_table["condition"].map(condition_order)
df_table = df_table.sort_values(["_mr", "_cr"]).drop(columns=["_mr", "_cr", "model"])

print(f"Full metric table: {df_table.shape}")
display(df_table.round(2))

# ── Summary statistics per dagspace ──
for ds_name, prefix in sorted(DS_PREFIX.items()):
    ds_cols = [c for c in df_table.columns
              if c.startswith(f"{prefix}/") or c.startswith(f"{prefix}_")]
    if not ds_cols:
        continue
    sub = df_table[ds_cols]
    stats = sub.describe().T[["count", "mean", "std", "min", "50%", "max"]]
    stats["cv"] = (stats["std"] / stats["mean"].abs()).round(3)
    stats["spread"] = stats["max"] - stats["min"]
    print(f"\n{'=' * 80}")
    print(f"{ds_name}  ({len(ds_cols)} metrics)")
    print(f"{'=' * 80}")
    with pd.option_context("display.max_rows", None, "display.float_format", "{:.2f}".format):
        display(stats.sort_values("spread", ascending=False))


## 4. Compute deltas (Instruct − No-FT)

For each model + metric, compute the absolute change from no-FT → instruct.

In [ ]:
# Metrics to include in the delta table / LaTeX / plots
# (df_column, display_label, lower_is_better)
PAPER_METRICS = [
    ("gc_applicability_f1",      "App F1",          False),
    ("gc_compliance_f1",         "Comp F1",         False),
    ("pl/qa_accuracy",           "QA Acc",          False),
    ("pl/adjusted_leakage_rate", "Adj Leak",        True),
    ("pl/helpful_rate",          "Helpful",         False),
    ("ca/pearson_r",             "Pearson r",       False),
    ("cirl/complete",            "Complete",        False),
    ("vlm/Q7/accuracy",          "Q7 Acc",          False),
]
PAPER_METRICS = [(col, lbl, lb) for col, lbl, lb in PAPER_METRICS
                 if col in df_table.columns]

# Pivot wide: one row per model, columns = (metric, condition)
metric_col_names = [col for col, _, _ in PAPER_METRICS]
df_wide = df_table.pivot_table(index="Model", columns="condition", values=metric_col_names)

# Build delta table
delta_rows = []
for model in MODEL_ORDER:
    if model not in df_wide.index:
        continue
    row = {"Model": model}
    for col, lbl, _lb in PAPER_METRICS:
        noft = df_wide.loc[model].get((col, "no-ft"), np.nan)
        inst = df_wide.loc[model].get((col, "instruct"), np.nan)
        row[f"{lbl} (NoFT)"] = noft
        row[f"{lbl} (Instr)"] = inst
        row[f"{lbl} (Δ)"] = (inst - noft) if pd.notna(noft) and pd.notna(inst) else np.nan
    delta_rows.append(row)

df_delta = pd.DataFrame(delta_rows).set_index("Model")
display(df_delta.round(2))


## 5. Generate LaTeX table

Grouped rows per model: no-FT row, instruct row, delta row. Benchmark columns grouped with `\cmidrule`.

In [ ]:
def build_comparison_latex(df_delta: pd.DataFrame) -> str:
    """Generate a booktabs LaTeX table comparing no-FT vs instruct."""
    # Group metrics by benchmark for header spans
    bench_groups = [
        ("GoldCoin-HIPAA", ["App F1", "Comp F1"]),
        ("PrivacyLens",    ["QA Acc", "Adj Leak", "Helpful"]),
        ("ConfAIde",       ["Pearson r"]),
        ("CIRL-Vig.",      ["Complete"]),
        ("VLM-GeoPri.",    ["Q7 Acc"]),
    ]
    all_metric_names = [lbl for _, lbl, _ in PAPER_METRICS
                        if f"{lbl} (NoFT)" in df_delta.columns]
    lower_better = {lbl for _, lbl, lb in PAPER_METRICS if lb}

    lines = []
    lines.append(r"\begin{table}[t]")
    lines.append(r"\centering")
    lines.append(r"\caption{Pretrained (no-FT) vs.\ instruction-tuned performance across CI benchmarks (\%). "
                 r"Adj Leak = leakage among helpful responses only. "
                 r"$\Delta$ = instruct $-$ no-FT. $\downarrow$ = lower is better.}")
    lines.append(r"\label{tab:instruct-vs-no-ft}")
    n_data_cols = len(all_metric_names)
    lines.append(r"\begin{tabular}{ll" + "c" * n_data_cols + "}")
    lines.append(r"\toprule")

    # Header row 1: benchmark group spans
    group_header = " & "
    col_idx = 3
    cmidrules = []
    for bench_name, bench_cols in bench_groups:
        present = [c for c in bench_cols if c in all_metric_names]
        if present:
            n = len(present)
            group_header += rf" & \multicolumn{{{n}}}{{c}}{{\textbf{{{bench_name}}}}}"
            cmidrules.append(rf"\cmidrule(lr){{{col_idx}-{col_idx + n - 1}}}")
            col_idx += n
    lines.append(group_header + r" \\")
    lines.append(" ".join(cmidrules))

    # Header row 2: metric names
    col_labels = {
        "Adj Leak": r"Adj Leak $\downarrow$",
        "Pearson r": r"Pearson $r$",
    }
    col_headers = [r"\textbf{Model}", r"\textbf{Cond.}"] + [
        col_labels.get(c, c) for c in all_metric_names
    ]
    lines.append(" & ".join(col_headers) + r" \\")
    lines.append(r"\midrule")

    # Body: one group per model (no-FT, instruct, delta)
    for i, (model_name, row) in enumerate(df_delta.iterrows()):
        name_str = rf"\textbf{{{model_name}}}" if model_name in BOLD_MODELS else str(model_name)

        # No-FT row
        noft_cells = [name_str, "No-FT"]
        for col in all_metric_names:
            val = row.get(f"{col} (NoFT)", np.nan)
            noft_cells.append("---" if pd.isna(val) else f"{val:.1f}")
        lines.append(" & ".join(noft_cells) + r" \\")

        # Instruct row
        inst_cells = ["", "Instruct"]
        for col in all_metric_names:
            val = row.get(f"{col} (Instr)", np.nan)
            inst_cells.append("---" if pd.isna(val) else f"{val:.1f}")
        lines.append(" & ".join(inst_cells) + r" \\")

        # Delta row
        delta_cells = ["", r"$\Delta$"]
        for col in all_metric_names:
            val = row.get(f"{col} (Δ)", np.nan)
            if pd.isna(val):
                delta_cells.append("---")
            else:
                sign = "+" if val > 0 else ""
                txt = f"{sign}{val:.1f}"
                improved = (val < 0) if col in lower_better else (val > 0)
                if improved:
                    txt = rf"\textcolor{{teal}}{{{txt}}}"
                elif val != 0:
                    txt = rf"\textcolor{{red}}{{{txt}}}"
                delta_cells.append(txt)
        lines.append(" & ".join(delta_cells) + r" \\")

        if i < len(df_delta) - 1:
            lines.append(r"\addlinespace")

    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


latex = build_comparison_latex(df_delta)
print(latex)


## 6. Save LaTeX table

In [ ]:
out_dir = Path("tables")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "instruct_v_no_ft.tex"
out_path.write_text(latex)
print(f"Saved to {out_path.resolve()}")

## 7. Comparative bar plots

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Plot style ─────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 7.5,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.alpha": 0.25,
    "grid.linewidth": 0.5,
})

# Colors for conditions
COND_COLORS = {"no-ft": "#4C72B0", "instruct": "#DD8452"}

In [ ]:
# ── Figure 1: Grouped bar chart per metric (no-FT vs instruct side by side) ──
from math import ceil

metric_spec = [(lbl, lbl, not lb) for _, lbl, lb in PAPER_METRICS
               if f"{lbl} (NoFT)" in df_delta.columns]

# Display labels for plot titles
PLOT_TITLES = {
    "App F1": "GoldCoin Appl. F1 (%)",
    "Comp F1": "GoldCoin Comp. F1 (%)",
    "QA Acc": "PrivacyLens QA Acc (%)",
    "Adj Leak": "PrivacyLens Adj Leak Rate (%)",
    "Helpful": "PrivacyLens Helpful Rate (%)",
    "Pearson r": "ConfAIde Pearson r (%)",
    "Complete": "CIRL Completeness (%)",
    "Q7 Acc": "VLM-GeoPrivacy Q7 Acc (%)",
}

models = list(df_delta.index)
n_models = len(models)
n_metrics = len(metric_spec)
bar_w = 0.4
x = np.arange(n_models)

ncols = ceil(n_metrics / 2)
fig, axes = plt.subplots(2, ncols, figsize=(5 * ncols, 10), squeeze=False)
axes = axes.flatten()

for ax, (col, _lbl, higher_better) in zip(axes, metric_spec):
    noft_vals = df_delta[f"{col} (NoFT)"].reindex(models).values
    inst_vals = df_delta[f"{col} (Instr)"].reindex(models).values

    ax.bar(x - bar_w/2, noft_vals, bar_w, color=COND_COLORS["no-ft"],
           label="No-FT", edgecolor="white", linewidth=0.5, zorder=3)
    ax.bar(x + bar_w/2, inst_vals, bar_w, color=COND_COLORS["instruct"],
           label="Instruct", edgecolor="white", linewidth=0.5, zorder=3)

    # Star on best bar (across all models × both conditions)
    best_val, best_x, best_color = None, None, None
    for idx, (nv, iv) in enumerate(zip(noft_vals, inst_vals)):
        for v, xpos, cond in [(nv, x[idx] - bar_w/2, "no-ft"),
                               (iv, x[idx] + bar_w/2, "instruct")]:
            if pd.notna(v):
                is_better = (best_val is None
                             or (not higher_better and v < best_val)
                             or (higher_better and v > best_val))
                if is_better:
                    best_val, best_x, best_color = v, xpos, COND_COLORS[cond]
    if best_val is not None:
        ax.annotate("$\\bigstar$",
                    xy=(best_x, best_val),
                    xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=14, color=best_color, zorder=5)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"$\\bf{{{m}}}$" if m in BOLD_MODELS else m for m in models],
        rotation=35, ha="right", fontsize=14)
    suffix = "" if higher_better else r" ($\downarrow$)"
    ax.set_title(PLOT_TITLES.get(col, col) + suffix,
                fontsize=16, fontweight="bold", y=1.1)

    ax.set_ylim(0, 105)

# Figure-wide legend, centered above the top row of panels
handles, labels = axes[0].get_legend_handles_labels()
by_label = dict(zip(labels, handles))
fig.legend(by_label.values(), by_label.keys(),
           loc="upper center", bbox_to_anchor=(0.5, 1.07),
           ncol=len(by_label), framealpha=0.95, fontsize=16,
           handlelength=1.5, handletextpad=0.5, columnspacing=2.0,
           borderpad=0.6, edgecolor="#cccccc")
# Remove any per-axis legends
for ax in axes[:n_metrics]:
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
for ax in axes[n_metrics:]:
    ax.axis("off")

plt.tight_layout()
fig.savefig("tables/instruct_v_no_ft_per_metric.pdf")
plt.show()
